# Module 04 — Saved Objects Deep Dive

The Kibana **feature state** in a snapshot captures every Kibana saved object.
This module walks through each saved object type, showing:

1. **What it is** and when you create it
2. **How to create it** via the Kibana UI (step-by-step) and confirm it exists via the API
3. **The saved object schema** — the actual JSON stored in `.kibana*`
4. **Snapshot → delete → restore cycle** to prove the object is saved and recoverable

## Saved object types covered

| # | Type | Description |
|---|------|-------------|
| 4.1 | `index-pattern` | Data Views — the lens through which Kibana reads ES data |
| 4.2 | `search` | Saved Searches from Discover |
| 4.3 | `visualization` | Legacy Aggregation-based visualizations |
| 4.4 | `lens` | Lens visualizations |
| 4.5 | `map` | Maps visualizations |
| 4.6 | `dashboard` | Dashboards and their panel reference graph |
| 4.7 | `canvas-workpad` | Canvas workpads |
| 4.8 | `tag` | Tags applied to saved objects |
| 4.9 | `query` | Saved KQL/Lucene queries |
| 4.10 | `space` | Kibana Spaces |
| 4.11 | `alert` | Alerting Rules |
| 4.12 | `action` | Connectors |
| 4.13 | `cases` + `cases-comments` + `cases-user-actions` | Cases |
| 4.14 | `config` | Kibana configuration |
| 4.15 | `short-url` | Short URLs |
| 4.16 | `event-annotation-group` | Event annotation groups |
| 4.17 | — | Cross-type restore ordering |

In [ ]:
import json, uuid, time
from helpers import *

es = get_client()
wait_for_green(es)
register_fs_repo(es)


: 

---
## 4.1 Data Views (`index-pattern`)

**Data Views** (formerly called Index Patterns) are the foundation of Kibana's data layer.
Every time Kibana queries Elasticsearch — to draw a chart, populate Discover, or run an
alert — it does so through a Data View. The Data View tells Kibana which index pattern to
target (wildcards allowed, e.g. `logs-*`), which field holds the timestamp, and how each
field should be displayed or formatted.

Because virtually every other saved object ultimately references a Data View (searches,
visualizations, Lens charts, maps, dashboards, alerts all have at least one
`index-pattern` in their `references` array), **Data Views are the single most important
type to get right when restoring**. If a Data View is missing after a restore the objects
that reference it will silently break — charts will refuse to load and Discover will show
an error. Kibana's restore ordering ensures Data Views are always created before anything
that depends on them.

Key attributes stored in the saved object:
- `title` — the index pattern glob (e.g. `kibana_sample_data_ecommerce`)
- `timeFieldName` — the default time field used by all time-series charts
- `fieldFormatMap` — per-field display overrides (e.g. currency symbol for a price field)
- `runtimeFieldMap` — Painless-based runtime fields defined at the Kibana layer

### Create in Kibana UI
1. Go to **Stack Management → Data Views**
2. Click **Create data view**
3. Enter pattern `training-orders-*`, name it `Training Orders`
4. Select `@timestamp` as the time field
5. Click **Save data view to Kibana**

We will also create one via the API for the exercise:

In [ ]:
heading('4.1 Data View — create')

dv_response = kibana_post('/api/data_views/data_view', {
    'data_view': {
        'title': 'training-orders-*',
        'name': 'Training Orders',
        'timeFieldName': '@timestamp',
    },
})
dv_id = dv_response['data_view']['id']
success(f'Data view created: id={dv_id}')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
# The data view we just created should appear here:
kibana_link('/app/management/kibana/dataViews', 'Stack Management → Data Views')

In [ ]:
# Inspect the saved object schema
heading('Data View saved object schema')
obj_list = find_saved_objects('index-pattern')
training_dv = next((o for o in obj_list if o['attributes'].get('name') == 'Training Orders'), None)
if training_dv:
    pp(training_dv, 'index-pattern saved object')
    console.print('')
    info('Key fields in attributes:')
    info('  title           — the index pattern string (wildcards allowed)')
    info('  timeFieldName   — default time field')
    info('  fields          — JSON string: field type overrides')
    info('  fieldFormatMap  — JSON string: display format per field')
    info('  runtimeFieldMap — runtime fields defined in this data view')

In [ ]:
# Snapshot → delete → restore
snap_delete_restore_cycle('snap-dv-test', 'Data View')

# Delete it
kibana_delete(f'/api/data_views/data_view/{dv_id}')
warn(f'Deleted data view {dv_id}')
assert not any(o['id'] == dv_id for o in find_saved_objects('index-pattern')), 'Should be gone'

# Restore
restore_kibana_state(es, SNAP_REPO, 'snap-dv-test')
time.sleep(3)  # allow Kibana to process restored objects

# Confirm
restored = find_saved_objects('index-pattern')
assert any(o['id'] == dv_id for o in restored), 'Data view should be restored'
success(f'Data view {dv_id} successfully restored!')

---
## 4.2 Saved Searches (`search`)

A saved search captures the full state of a **Discover session**: the KQL/Lucene query,
any pinned filters, the selected columns, the sort order, and the surrounding data view.
You can think of it as a reusable "view" into an index — a starting point you can share
with a colleague or embed as a panel inside a dashboard.

The heart of the schema is the `kibanaSavedObjectMeta.searchSourceJSON` field — a JSON
blob stored *inside* the JSON of the saved object. It contains the query, filter array,
and an `indexRefName` string that acts as a pointer name resolved against the `references`
array at load time. This indirection is how Kibana keeps the saved search portable:
the actual Data View ID is stored in `references`, not hard-coded into the query blob.

From a restore perspective, saved searches depend on exactly one Data View. If the
referenced Data View is absent when the search is loaded, Kibana will show an error.
During a feature-state restore this is handled automatically because Data Views are
restored first.

### Create in Kibana UI
1. Go to **Discover**
2. Select the `Kibana Sample Data eCommerce` data view
3. Set a KQL filter: `category : "Men's Clothing"`
4. Add columns: `customer_full_name`, `taxful_total_price`, `order_date`
5. Click **Save** → name it `Men's Clothing Orders`

In [ ]:
heading('4.2 Saved Search — inspect existing (loaded with sample data)')

# The eCommerce sample dataset creates saved searches automatically
searches = find_saved_objects('search')
console.print(f'  Found {len(searches)} saved search(es)')
if searches:
    pp(searches[0], 'search saved object')
    info('Key fields in attributes:')
    info('  title              — display name')
    info('  kibanaSavedObjectMeta.searchSourceJSON — query, filter, index (data view ref)')
    info('  columns            — selected Discover columns')
    info('  sort               — default sort [field, direction]')
    info('references           — array linking to the data view (index-pattern) this search uses')

In [ ]:
# Create a saved search via API
heading('Create saved search via API')

# First get the ecommerce data view id
dv_list = find_saved_objects('index-pattern')
ecom_dv = next((o for o in dv_list if 'ecommerce' in o['attributes'].get('title', '')), None)

if ecom_dv:
    ecom_dv_id = ecom_dv['id']
    search_resp = kibana_post('/api/saved_objects/search', {
        'attributes': {
            'title': "Men's Clothing Orders",
            'columns': ['customer_full_name', 'taxful_total_price', 'order_date'],
            'sort': [['order_date', 'desc']],
            'kibanaSavedObjectMeta': {
                'searchSourceJSON': json.dumps({
                    'query': {'query': "category : \"Men's Clothing\"", 'language': 'kuery'},
                    'filter': [],
                    'indexRefName': 'kibanaSavedObjectMeta.searchSourceJSON.index',
                }),
            },
        },
        'references': [{'name': 'kibanaSavedObjectMeta.searchSourceJSON.index', 'type': 'index-pattern', 'id': ecom_dv_id}],
    })
    search_id = search_resp['id']
    success(f"Saved search created: id={search_id}")

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
# Open Discover to see saved searches and apply the one we just created:
kibana_link('/app/discover', 'Discover — open saved search from the list')

In [ ]:
# Snapshot → delete → restore
snap_delete_restore_cycle('snap-search-test', 'Saved Search')

kibana_delete(f'/api/saved_objects/search/{search_id}')
warn(f'Deleted saved search {search_id}')

restore_kibana_state(es, SNAP_REPO, 'snap-search-test')
time.sleep(3)

restored = find_saved_objects('search')
assert any(o['id'] == search_id for o in restored)
success('Saved search restored!')

---
## 4.3 Legacy Visualizations (`visualization`)

The `visualization` type pre-dates Lens. It covers **Aggregation-based charts** (bar,
line, pie, data table, metric, gauge, goal), **TSVB** (Time Series Visual Builder),
**Timelion**, and **Vega/Vega-Lite** visualizations. All of these still use the `visualization`
saved object type — even in modern Kibana versions.

The distinguishing feature of the schema is the `visState` attribute: a JSON string
(JSON embedded inside JSON) that carries the entire visualization definition — the
`type` field tells Kibana which renderer to use, and `params` contains the
renderer-specific configuration. Because `visState` is opaque from Kibana's
perspective, its structure varies wildly between chart types. TSVB stores time-series
panel configs; Vega stores raw Vega-Lite specs; aggregation charts store a `aggs` array.

For snapshot purposes, legacy visualizations behave like any other saved object — they
are fully captured in the Kibana feature state. The one subtlety is that TSVB and
aggregation-based charts reference a Data View via the `references` array, while Vega
charts that embed their own ES queries may not — making Vega workpads closer to
Canvas workpads in their self-contained portability.

### Create in Kibana UI
1. Go to **Visualize Library → Create visualization**
2. Choose **Aggregation based → Vertical bar**
3. Select the eCommerce data view
4. Y-axis: `Sum` of `taxful_total_price`
5. X-axis: `Date histogram` on `order_date`
6. Save as `Training — Revenue by Day`

In [ ]:
heading('4.3 Legacy Visualization — inspect sample data objects')

vizzes = find_saved_objects('visualization')
console.print(f'  Found {len(vizzes)} visualization(s)')
if vizzes:
    v = vizzes[0]
    pp(v, 'visualization saved object')
    info('Key fields in attributes:')
    info('  visState        — JSON blob: vis type, params, aggs')
    info('  uiStateJSON     — panel-level UI state')
    info('  kibanaSavedObjectMeta.searchSourceJSON — linked data view')
    info('references       — links to index-pattern and any saved search')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/visualize', 'Visualize Library — browse legacy visualizations')

---
## 4.4 Lens Visualizations (`lens`)

Lens is the **modern, drag-and-drop** visualization builder introduced in Kibana 7.5.
It replaced the older aggregation-based flow for most chart types and is the recommended
path for all new visualizations. Unlike legacy visualizations, Lens always uses the `lens`
saved object type, so the two are easy to distinguish in the API.

The schema is considerably richer than the `visualization` type. Instead of an opaque
`visState` blob, Lens stores a structured `state` object with:
- `visualization` — the chart type and its per-layer rendering configuration
- `query` — a global filter applied on top of the layer queries
- `filters` — pinned filters from the Kibana filter bar
- `datasourceStates.formBased.layers` — each layer's field selections, aggregations,
  and column formulas

The `references` array links each layer to its Data View by ID.  Because the structure
is machine-readable, Kibana can migrate Lens objects between versions more reliably
than it can migrate `visState` blobs.

### Create in Kibana UI
1. Go to **Visualize Library → Create visualization → Lens**
2. Select the eCommerce data view
3. Drag `taxful_total_price` to the Y-axis (it auto-suggests Sum)
4. Drag `order_date` to the X-axis (it auto-suggests Date histogram)
5. Save as `Training — Lens Revenue Chart`

In [ ]:
heading('4.4 Lens Visualization — inspect sample data objects')

lens_objs = find_saved_objects('lens')
console.print(f'  Found {len(lens_objs)} lens object(s)')
if lens_objs:
    l = lens_objs[0]
    pp(l, 'lens saved object')
    info('Key fields:')
    info('  state.visualization  — chart type + layer configs')
    info('  state.datasourceStates.formBased.layers — field + operation definitions per layer')
    info('  state.filters        — global filters applied to this Lens')
    info('  state.query          — KQL query')
    info('references             — data view IDs used by each layer')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/visualize', 'Visualize Library — browse Lens visualizations')

---
## 4.5 Maps (`map`)

The Maps application stores geospatial visualizations as `map` saved objects. A map can
contain multiple **layers** — Documents (scatter of ES documents), Heat maps, Clusters,
EMS boundaries (country/region outlines from Elastic Maps Service), and Grid aggregations.
Each layer has its own data source configuration, style rules, and visibility filters.

The schema stores layers as `layerListJSON` (another JSON-in-JSON pattern), with each
entry carrying a `sourceDescriptor` that describes where the geo data comes from.
Document layers reference a Data View and a geo field; EMS layers reference an external
tile manifest. This means a map with EMS layers has no `references` entries at all —
it is self-contained — while a Document layer produces a reference to the Data View.

For restore purposes, maps behave like Lens charts: they are fully captured in the
`kibana` feature state, they restore cleanly as long as the referenced Data Views exist,
and they migrate well between patch versions. They do *not* snapshot the underlying
geo index data — that is captured separately as a normal index snapshot.

### Create in Kibana UI
1. Go to **Maps → Create map**
2. Click **Add layer → Documents**
3. Select the eCommerce data view and the `geoip.location` field
4. Save as `Training — Customer Locations`

In [ ]:
heading('4.5 Maps — inspect saved objects')

maps = find_saved_objects('map')
console.print(f'  Found {len(maps)} map(s)')
if maps:
    m = maps[0]
    # Maps can be large; show structure only
    attrs = m['attributes']
    console.print(f"  Title: {attrs.get('title')}")
    try:
        map_state = json.loads(attrs.get('mapStateJSON', '{}'))
        pp(map_state, 'mapStateJSON')
    except Exception:
        pass
    info('Key fields:')
    info('  layerListJSON   — array of layer descriptors (type, source, style)')
    info('  mapStateJSON    — zoom, center coordinates, timeFilters')
    info('  uiStateJSON     — layer visibility state')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/maps', 'Maps — view and edit map saved objects')

---
## 4.6 Dashboards (`dashboard`)

A dashboard is a **container of panels**, not a standalone visualization. Each panel on a
dashboard is a reference to another saved object — a Lens chart, a legacy visualization,
a Map, a saved search, or a Markdown widget — plus layout metadata (position, size) stored
in `panelsJSON`. This makes the dashboard object itself relatively lightweight; it delegates
all the rendering logic to the objects it references.

The `references` array is the key to understanding dashboard restore. Every panel that
points to an external saved object appears as a reference entry. When Kibana restores a
feature state snapshot, it must restore the referenced objects *before* the dashboard,
otherwise the panels will render as "object not found" errors. The feature-state mechanism
handles this ordering automatically.

A common pitfall in partial restores: if you restore only the dashboard object (e.g. by
importing it via the Saved Objects UI without its dependencies), you end up with a
dashboard that renders correctly for objects still present in Kibana but shows errors for
any panel whose source was deleted or never created. Always use the full feature-state
restore — or the "include related objects" option in the Saved Objects import UI — to
keep the graph intact.

Controls (time slider, option lists, range sliders) added to a dashboard are stored
inline in the dashboard's own JSON since Kibana 8.3, rather than as separate saved
objects, which simplifies the reference graph.

### Create in Kibana UI
1. Go to **Dashboards → Create dashboard**
2. Click **Add from library** and add the eCommerce visualizations
3. Arrange panels and save as `Training — eCommerce Overview`

In [ ]:
heading('4.6 Dashboard — inspect saved objects')

dashboards = find_saved_objects('dashboard')
console.print(f'  Found {len(dashboards)} dashboard(s)')
if dashboards:
    d = dashboards[0]
    attrs = d['attributes']
    console.print(f"  Title: {attrs.get('title')}")

    try:
        panels = json.loads(attrs.get('panelsJSON', '[]'))
        console.print(f'  Panels: {len(panels)}')
        for p in panels[:3]:
            console.print(f"    panel {p.get('panelIndex')}: type={p.get('type')}  embeddableConfig={list(p.get('embeddableConfig', {}).keys())}")
    except Exception:
        pass

    console.print(f'  References: {len(d["references"])} linked objects')
    for ref in d['references'][:5]:
        console.print(f"    {ref['type']} : {ref['id']} (alias={ref['name']})")

    info('Key fields:')
    info('  panelsJSON      — layout + embeddable config for each panel')
    info('  optionsJSON     — use margins, sync colors, sync cursor, etc.')
    info('  kibanaSavedObjectMeta.searchSourceJSON — dashboard-level filter')
    info('references        — the critical link graph: panels → vizzes → data views')

In [ ]:
# Visualise the reference graph
heading('Dashboard reference graph')

if dashboards:
    d = dashboards[0]
    from rich.tree import Tree
    tree = Tree(f"[bold cyan]dashboard[/bold cyan] → {d['attributes']['title']} ({d['id'][:8]}...)")
    for ref in d['references']:
        tree.add(f"[yellow]{ref['type']}[/yellow] → {ref['id'][:8]}...  ({ref['name']})")
    console.print(tree)

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
# Open the dashboard list, then click any dashboard to inspect its panels:
kibana_link('/app/dashboards', 'Dashboards — open the dashboard list')
# Or jump directly to the first sample dashboard:
if dashboards:
    d = dashboards[0]
    kibana_link(f'/app/dashboards#/view/{d["id"]}',
                f'Open dashboard: {d["attributes"]["title"]}')

In [ ]:
# Snapshot the entire Kibana state, delete ALL sample dashboards, restore, confirm
heading('Dashboard snapshot → delete → restore cycle')

snap_delete_restore_cycle('snap-dash-test', 'Dashboard')

dash_ids = [d['id'] for d in dashboards]
for did in dash_ids:
    try:
        kibana_delete(f'/api/saved_objects/dashboard/{did}')
    except Exception:
        pass
warn(f'Deleted {len(dash_ids)} dashboard(s)')

restore_kibana_state(es, SNAP_REPO, 'snap-dash-test')
time.sleep(4)

restored_dashes = find_saved_objects('dashboard')
success(f'Restored {len(restored_dashes)} dashboard(s) — original count: {len(dash_ids)}')

---
## 4.7 Canvas Workpads (`canvas-workpad`)

Canvas is Kibana's **presentation layer** — it produces pixel-perfect, brand-able reports
and infographics driven by live Elasticsearch data. A workpad is a collection of pages;
each page contains elements (text, images, metrics, charts) whose data is fetched via
**Canvas expressions** embedded directly in the element definition.

The `canvas-workpad` saved object is unique among Kibana types because it has **no
external `references` array**. All data-fetching logic — the ES|QL queries, SQL
expressions, or `esaggs` calls — is embedded inline inside `pages[].elements[].expression`.
This makes workpads fully self-contained: you can export a single workpad object and it
carries everything it needs to run on any cluster, as long as the target indices exist.

From a restore perspective this is both a blessing and a constraint. The blessing: you
can restore just the workpad without worrying about dependency ordering. The constraint:
if the underlying index schema changes between snapshot and restore (different field names,
for example), the embedded expressions will break silently because Kibana has no way to
validate the field references at restore time.

### Create in Kibana UI
1. Go to **Canvas**
2. Click **Create workpad**
3. Add a metric element, configure with an ES|QL expression
4. Save (Canvas auto-saves on changes)

In [ ]:
heading('4.7 Canvas Workpad')

workpads = find_saved_objects('canvas-workpad')
console.print(f'  Found {len(workpads)} canvas workpad(s)')
if workpads:
    w = workpads[0]
    attrs = w['attributes']
    console.print(f"  Name    : {attrs.get('name')}")
    console.print(f"  Width   : {attrs.get('width')}, Height: {attrs.get('height')}")
    pages = attrs.get('pages', [])
    console.print(f"  Pages   : {len(pages)}")
    info('Key fields:')
    info('  pages[]         — array of page objects, each with elements[]')
    info('  elements[].expression — Canvas expression (essql, esaggs, esdocs, etc.)')
    info('  No external references — workpad is entirely self-contained')
else:
    info('No canvas workpads found — create one in the Kibana UI and re-run this cell.')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/canvas', 'Canvas — view workpads')

---
## 4.8 Tags (`tag`)

Tags are lightweight labels — a name, a colour, and an optional description — that you
can attach to any saved object for organisation and filtering in the Saved Objects UI and
in Kibana's global search bar.

What makes tags interesting from a schema perspective is the **reference direction**.
You might expect the tag to hold a list of objects it has been applied to, but that is
not how it works. Instead, *the tagged objects hold a reference pointing back to the tag*.
A dashboard with two tags will have two entries in its `references` array of type `tag`.
The tag object itself is tiny and has no references of its own.

This matters at restore time: restoring a tagged dashboard without its referenced tags
will succeed (Kibana tolerates missing tag references), but the tags won't appear in the
filter UI. A complete feature-state restore brings back both tags and their tagged objects
simultaneously, so the relationship is always intact.

Tags are global within a Kibana Space — any object in the space can reference any tag in
the same space. They are not shared across Spaces.

In [ ]:
heading('4.8 Tags — create and apply')

tag_resp = kibana_post('/api/saved_objects/tag', {
    'attributes': {
        'name': 'training',
        'description': 'Created by snapshot training module',
        'color': '#007bff',
    },
})
tag_id = tag_resp['id']
success(f'Tag created: id={tag_id}')

pp(tag_resp, 'tag saved object')
info('Key fields:')
info('  name        — display label')
info('  color       — hex color for the badge')
info('  description — tooltip text')
info('  Tags are referenced FROM tagged objects — the tag itself has no references array')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/management/kibana/tags', 'Stack Management → Tags')

In [ ]:
# Apply the tag to an existing dashboard (if any)
dashboards = find_saved_objects('dashboard')
if dashboards:
    d = dashboards[0]
    existing_refs = d.get('references', [])
    # Update the dashboard's references to include the tag
    new_refs = existing_refs + [{'type': 'tag', 'id': tag_id, 'name': f'tag-{tag_id}'}]
    kibana_post(f'/api/saved_objects/dashboard/{d["id"]}?overwrite=true', {
        'attributes': d['attributes'],
        'references': new_refs,
    })
    success(f"Tag applied to dashboard '{d['attributes']['title']}'")

    # Confirm the reference exists
    updated = kibana_get(f"/api/saved_objects/dashboard/{d['id']}")
    tag_refs = [r for r in updated.get('references', []) if r['type'] == 'tag']
    console.print(f'  Tag references on dashboard: {tag_refs}')

---
## 4.9 Saved Queries (`query`)

A saved query stores a **reusable KQL or Lucene expression** — and optionally a set of
pinned filters and a time range — that appears in the search bar drop-down in Discover,
Dashboards, and the Alerting rule editor. Think of it as a named shortcut for a frequently
used search predicate.

Unlike a saved search, a saved query is *not* tied to a specific Data View. Its schema is
minimal: `title`, `description`, `query.query` (the expression string), `query.language`
(`kuery` or `lucene`), `filters[]`, and an optional `timefilter` with `from`/`to`/`refreshInterval`.
Because there is no `references` array, saved queries have no dependencies and are among
the simplest objects to restore.

The practical impact during a disaster-recovery scenario is that saved queries let your
analysts get back to work quickly — once Kibana is restored, every named search expression
they rely on in the search bar is immediately available again, without anyone needing to
remember and retype complex KQL predicates.

In [ ]:
heading('4.9 Saved Query — create')

sq_resp = kibana_post('/api/saved_queries/_create', {
    'title': 'High value orders',
    'description': 'Orders over $100',
    'query': {
        'query': 'taxful_total_price > 100',
        'language': 'kuery',
    },
    'filters': [],
    'timefilter': {
        'from': 'now-30d',
        'to': 'now',
        'refreshInterval': {'pause': True, 'value': 0},
    },
})
sq_id = sq_resp['id']
success(f'Saved query created: id={sq_id}')
pp(sq_resp, 'query saved object')
info('Key fields: title, description, query.query, query.language, filters[], timefilter')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
# Saved queries appear in the KQL search bar drop-down in Discover:
kibana_link('/app/discover', 'Discover — click the saved query icon in the search bar')

In [ ]:
# Snapshot → delete → restore
snap_delete_restore_cycle('snap-query-test', 'Saved Query')

requests_mod = __import__('requests')
requests_mod.delete(
    f'{KIBANA_HOST}/api/saved_queries/{sq_id}',
    auth=('elastic', ELASTIC_PASSWORD),
    headers={'kbn-xsrf': 'true'},
)
warn(f'Deleted saved query {sq_id}')

restore_kibana_state(es, SNAP_REPO, 'snap-query-test')
time.sleep(3)

queries_resp = kibana_get(f'/api/saved_queries/_all')
restored_ids = [q['id'] for q in queries_resp.get('savedQueries', [])]
assert sq_id in restored_ids
success('Saved query restored!')

---
## 4.10 Kibana Spaces (`space`)

Spaces are Kibana's **multi-tenancy mechanism**: each space is an isolated namespace with
its own set of saved objects. A dashboard in Space A is invisible from Space B; a Data View
created in Space A does not exist in Space B. Users can be restricted to one or more spaces
via role-based access control.

Under the hood, every saved object that belongs to a non-default space has its Elasticsearch
document stored in a dedicated `.kibana_<space-id>` index (or a shared `.kibana` index with
a `namespaces` field, depending on whether the type uses the legacy or modern namespace
model). The `space` saved object itself — the metadata record for the space — is always
stored in the `default` space's `.kibana` index.

When you take a `kibana` feature-state snapshot, **every space is included**: all the
`.kibana*` indices are snapshotted as a unit. This is important for recovery scenarios —
you do not need to snapshot spaces individually. The flip side is that you cannot
selectively restore a single space from a feature-state snapshot without also restoring
all other spaces. If you need per-space portability, use the **Saved Objects UI import/export**
or the **Copy to Space** feature instead.

When you snapshot and restore the `kibana` feature state, spaces are restored first
(they have no dependencies), and then each space's objects are restored into it.

In [ ]:
heading('4.10 Spaces — create a second space')

import requests as req

# Create a new space
space_resp = req.post(
    f'{KIBANA_HOST}/api/spaces/space',
    auth=('elastic', ELASTIC_PASSWORD),
    headers={'kbn-xsrf': 'true', 'Content-Type': 'application/json'},
    json={
        'id': 'training-space',
        'name': 'Training Space',
        'description': 'Isolated space for snapshot training exercises',
        'color': '#AA4433',
        'initials': 'TS',
    },
)
if space_resp.status_code in (200, 409):
    success('Space created (or already exists): training-space')
else:
    warn(f'Space creation: {space_resp.status_code} {space_resp.text}')

In [ ]:
# List all spaces
spaces = kibana_get('/api/spaces/space')
t = Table('ID', 'Name', 'Description')
for s in spaces:
    t.add_row(s['id'], s['name'], s.get('description', '—')[:50])
console.print(t)

info('Spaces are stored as saved objects of type "space" in the default space.')
info('Objects created in training-space use path: /s/training-space/api/...')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/management/kibana/spaces', 'Stack Management → Spaces')
# Switch to the new training space using the space selector (top-left avatar):
kibana_link('/s/training-space/app/home', 'Training Space — home page')

In [ ]:
# Create a data view inside the new space
heading('Create a data view inside training-space')

space_dv = kibana_post('/s/training-space/api/data_views/data_view', {
    'data_view': {
        'title': 'training-space-data-*',
        'name': 'Training Space Data View',
    },
})
space_dv_id = space_dv['data_view']['id']
success(f'Data view created in training-space: {space_dv_id}')

# Show it is NOT visible in the default space
default_dvs = find_saved_objects('index-pattern', space='default')
space_dvs = find_saved_objects('index-pattern', space='training-space')
console.print(f'  Data views in default space : {len(default_dvs)}')
console.print(f'  Data views in training-space: {len(space_dvs)}')

---
## 4.11 Alerting Rules (`alert`)

An alerting rule (saved as type `alert` — the UI renamed "alerts" to "rules" in Kibana 7.13
but the saved object type was never changed) periodically runs a **condition check** against
Elasticsearch and fires actions when that condition is met. The rule scheduler is driven by
the `schedule.interval` (or a cron expression) stored in the saved object.

The schema separates *what to check* from *where to send*:
- `params` contains the rule-type-specific query definition (e.g. ES|QL statement,
  threshold value, index pattern for an ES Query rule)
- `actions` contains an array of action definitions, each specifying a connector ID, a
  group (the trigger category), and message parameters
- `references` contains entries of type `action` pointing to the connector saved objects
  used in `actions`

Two things are **not** stored in the saved object:
- **Rule execution history** — past run results and alert states are stored in Elasticsearch
  system indices (`.kibana-event-log-*`, `.alerts-*`), not in the feature state. After a
  restore, rules restart with a clean execution history.
- **Rule scheduling state** — the rule will begin its schedule fresh from the restore point.

### Create in Kibana UI
1. Go to **Stack Management → Rules**
2. Click **Create rule**
3. Choose **Elasticsearch query** rule type
4. Configure: index=ecommerce, threshold: count > 1000 in 1 day
5. Add a server log action (connector)
6. Save as `Training — High Order Volume`

In [ ]:
heading('4.11 Alerting Rule — create via API')

rule_resp = kibana_post('/api/alerting/rule', {
    'name': 'Training — High Order Volume',
    'rule_type_id': '.es-query',
    'consumer': 'alerts',
    'schedule': {'interval': '1h'},
    'params': {
        'index': ['kibana_sample_data_ecommerce'],
        'timeField': 'order_date',
        'esQuery': json.dumps({'query': {'match_all': {}}}),
        'size': 100,
        'thresholdComparator': '>',
        'threshold': [500],
        'timeWindowSize': 24,
        'timeWindowUnit': 'h',
    },
    'actions': [],
    'notify_when': 'onActionGroupChange',
})
rule_id = rule_resp['id']
success(f'Rule created: id={rule_id}')

# Show the saved object schema
rules_so = find_saved_objects('alert')
training_rule = next((r for r in rules_so if r['id'] == rule_id), None)
if training_rule:
    pp(training_rule, 'alert saved object')
    info('Key fields in attributes:')
    info('  name            — display name')
    info('  rule_type_id    — identifies the rule type plugin (.es-query, etc.)')
    info('  schedule        — interval or cron expression')
    info('  params          — rule-type-specific parameters')
    info('  actions         — array of action objects (reference connectors via references[])')
    info('  notify_when     — onActiveAlert | onActionGroupChange | onThrottleInterval')
    info('  throttle        — minimum time between repeated alerts')
    info('  enabled         — whether the rule is active')
    info('references        — links to action (connector) saved objects')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link(
    '/app/management/insightsAndAlerting/triggersActions/rules',
    'Stack Management → Rules — view the rule we just created',
)
if 'rule_id' in dir():
    kibana_link(
        f'/app/management/insightsAndAlerting/triggersActions/rules/{rule_id}',
        f'Rule detail: Training — High Order Volume',
    )

---
## 4.12 Connectors (`action`)

Connectors (saved as type `action`) define the **delivery channel** for alerting actions:
Slack, email, PagerDuty, Jira, webhook, server log, and many others. A connector stores
connection details (URL, channel name, API endpoint) and credentials (API key, password,
OAuth token). Rules reference connectors via their `references` array; one connector can
be shared by many rules.

The critical thing to understand about connectors from a restore perspective is
**secret encryption**. Kibana encrypts the `secrets` field (passwords, API keys) at rest
using the `xpack.encryptedSavedObjects.encryptionKey` value set in `kibana.yml`. The
ciphertext is stored in the `.kibana` index and is included in a feature-state snapshot.

This creates an important cross-cluster restore constraint:
> If you restore a `kibana` feature state to a cluster whose Kibana uses a **different**
> `encryptedSavedObjects.encryptionKey`, the secrets will be unreadable. The connector
> objects will exist, but any rule that tries to fire an action will fail with a
> decryption error. You must re-enter the credentials for each affected connector.

To avoid this, ensure all target clusters use the same encryption key, or document
which connectors need their secrets re-entered as part of your DR runbook.

In [ ]:
heading('4.12 Connector — create server log connector')

connector_resp = kibana_post('/api/actions/connector', {
    'name': 'Training Server Log',
    'connector_type_id': '.server-log',
    'config': {},
    'secrets': {},
})
connector_id = connector_resp['id']
success(f'Connector created: id={connector_id}')

# Show the saved object
connectors_so = find_saved_objects('action')
training_connector = next((c for c in connectors_so if c['id'] == connector_id), None)
if training_connector:
    pp(training_connector, 'action (connector) saved object')
    info('Key fields:')
    info('  actionTypeId    — connector type identifier (.server-log, .slack, .email, etc.)')
    info('  config          — non-secret config (e.g. webhook URL, channel name)')
    info('  secrets         — encrypted credentials (not exposed in GET responses)')
    info('  NOTE: secrets are encrypted with the Kibana encryption key.')
    info('  If you restore to a cluster with a DIFFERENT encryption key, connectors with')
    info('  secrets will need their credentials re-entered after restore.')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link(
    '/app/management/insightsAndAlerting/triggersActions/connectors',
    'Stack Management → Connectors — view the server log connector',
)

In [ ]:
# Update the rule to reference the connector
heading('Link connector to rule')

kibana_post(f'/api/alerting/rule/{rule_id}', {
    'name': 'Training — High Order Volume',
    'schedule': {'interval': '1h'},
    'params': {
        'index': ['kibana_sample_data_ecommerce'],
        'timeField': 'order_date',
        'esQuery': json.dumps({'query': {'match_all': {}}}),
        'size': 100,
        'thresholdComparator': '>',
        'threshold': [500],
        'timeWindowSize': 24,
        'timeWindowUnit': 'h',
    },
    'actions': [{
        'id': connector_id,
        'group': 'query matched',
        'params': {'message': 'High order volume detected!', 'level': 'warn'},
    }],
    'notify_when': 'onActionGroupChange',
})

# Verify the reference
updated_rule_so = kibana_get(f'/api/saved_objects/alert/{rule_id}')
action_refs = [r for r in updated_rule_so.get('references', []) if r['type'] == 'action']
success(f'Rule now references connector(s): {action_refs}')

---
## 4.13 Cases (`cases`, `cases-comments`, `cases-user-actions`)

Kibana Cases provides a **lightweight ticketing and investigation workflow** built on top
of saved objects. It is used by the Security and Observability solutions to track incidents,
group related alerts, and maintain an audit trail of the investigation.

Every case generates three linked saved object types, each with a distinct role:

| Type | Description |
|------|-------------|
| `cases` | The case itself — title, description, status (`open`/`in-progress`/`closed`), severity, assignees, tags, and the external connector reference |
| `cases-comments` | Each comment or alert attachment. Comments are `user` type; alert attachments are `alert` type and carry a reference to the alert ID |
| `cases-user-actions` | An immutable audit log — one record per change to the case (status change, comment added, assignee updated, etc.) |

The three types are linked by `cases-comments` and `cases-user-actions` each holding a
reference to their parent `cases` object. This means restore order matters: the parent
`cases` object must be restored before its comments and user-action records. The
feature-state restore handles this automatically.

One gotcha: if a case references an external connector (e.g. Jira or ServiceNow) via the
`connector` field, and that connector's secrets are unreadable after a cross-cluster
restore (see section 4.12), the external sync for that case will stop working — but the
case data itself is fully intact.

In [ ]:
heading('4.13 Cases — create via API')

case_resp = kibana_post('/api/cases', {
    'title': 'Training Investigation Case',
    'description': 'Investigating unusual order volume spike',
    'tags': ['training', 'ecommerce'],
    'settings': {'syncAlerts': False},
    'severity': 'low',
    'connector': {'id': 'none', 'name': 'none', 'type': '.none', 'fields': None},
})
case_id = case_resp['id']
success(f'Case created: id={case_id}')
pp(case_resp, 'case object')

In [ ]:
# Add a comment
comment_resp = kibana_post(f'/api/cases/{case_id}/comments', {
    'type': 'user',
    'comment': 'Checked the ecommerce index — orders look anomalous between 2024-01-15 and 2024-01-17.',
    'owner': 'cases',
})
comment_id = comment_resp['id']
success(f'Comment added: id={comment_id}')

# Show the three related saved objects
for so_type in ['cases', 'cases-comments', 'cases-user-actions']:
    objs = find_saved_objects(so_type)
    console.print(f'  {so_type}: {len(objs)} object(s)')

info('All three types are included in the kibana feature state snapshot.')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
kibana_link('/app/cases', 'Cases — open the case we just created')
if 'case_id' in dir():
    kibana_link(f'/app/cases/{case_id}', f'Case detail: Training Investigation Case')

---
## 4.14 Configuration (`config`)

The `config` saved object is Kibana's **per-version, per-space settings store**. It
captures the values you set in **Stack Management → Advanced Settings**: the default Data
View, date and number formats, timezone, Dark Mode preference, and feature tour completion
flags, among others.

The object ID is the Kibana version string (e.g. `9.3.0`), which means each Kibana version
creates its own `config` object. When you upgrade Kibana, the new version reads the old
config object and migrates settings forward, then writes a new config object for the new
version. This is why after a major-version upgrade you may find multiple `config` objects
in the Saved Objects UI.

The most operationally significant attribute is `defaultIndex` — the ID of the Data View
that Kibana opens by default in Discover. If that Data View ID does not exist (e.g. after
a partial restore or a cluster migration), Discover will prompt the user to select a Data
View manually. Everything else in `config` is cosmetic and will degrade gracefully if
missing.

Because `config` is namespace-scoped, each Kibana Space has its own config object.
Restoring the full Kibana feature state brings back the config for every space.

In [ ]:
heading('4.14 Config saved object')

configs = find_saved_objects('config')
for c in configs:
    console.print(f"  id={c['id']}")
    pp(c['attributes'], 'config attributes')
    info('Key attributes:')
    info('  defaultIndex       — default data view ID (set when you star a data view)')
    info('  dateFormat         — display format for dates in Kibana')
    info('  theme:darkMode     — UI theme preference')
    info('  timepicker:*       — saved time picker defaults')
    info('  The config id is the Kibana version string (e.g. 9.3.0)')
    break

---
## 4.15 Short URLs (`short-url`)

When you click **Share → Get short URL** in Kibana, a `short-url` object is created that
maps a short hash (the **slug**) to a full application state URL. The browser navigates to
`/goto/<slug>` and Kibana redirects it to the full URL — which can encode dashboard panel
positions, time range, filter state, and other URL parameters that would otherwise produce
an unwieldy link.

Short URLs use the **Locator** system internally: instead of storing the raw URL, the
object stores a `locatorId` (e.g. `DASHBOARD_APP_LOCATOR`) and a `params` object
(e.g. `{dashboardId: "abc123"}`). Kibana resolves the actual URL from this at redirect
time. This makes short URLs more resilient to Kibana upgrades — the locator API can adapt
the params to the new URL structure without breaking every existing short URL.

The restore concern here is the **dangling reference** problem. A short URL points to a
specific dashboard (or Discover session, or map) by ID. If the target object was deleted
and not included in the restore, navigating to the short URL will produce a "not found"
error. Because short URLs are often embedded in external communication (emails, wiki pages,
dashboards in other tools), broken short URLs after a partial restore can be a significant
operational problem. This is one more reason to prefer full feature-state restores over
selective object restores.

In [ ]:
heading('4.15 Short URLs')

# Create a short URL via the API
dashboards = find_saved_objects('dashboard')
if dashboards:
    d = dashboards[0]
    short_url_resp = kibana_post('/api/short_url', {
        'locatorId': 'DASHBOARD_APP_LOCATOR',
        'params': {'dashboardId': d['id']},
    })
    short_id = short_url_resp.get('id')
    slug = short_url_resp.get('slug', '')
    success(f'Short URL created: id={short_id}  slug={slug}')
    pp(short_url_resp, 'short-url object')
    info('The short URL is resolved by Kibana at /goto/{slug}')

In [ ]:
# ── Kibana UI ────────────────────────────────────────────────────────────
# Short URLs redirect to the full dashboard URL in the browser.
# The short_url_resp above contains the 'slug' field — construct the redirect:
if 'short_url_resp' in dir() and short_url_resp:
    slug = short_url_resp.get('slug', '')
    kibana_link(f'/goto/{slug}', f'Short URL redirect → /goto/{slug}')

---
## 4.16 Event Annotation Groups (`event-annotation-group`)

Event annotations mark **specific time points** on Lens time series charts — as vertical
lines with optional labels and icons. They are useful for overlaying deployment events,
incidents, or business milestones on a metrics chart, so you can visually correlate a
spike in error rate with the deploy that caused it.

Annotations come in two flavours:
- **Manual** — a fixed timestamp with a label, stored entirely in the saved object
- **Query-based** — a KQL query run against an index; every matching document's timestamp
  becomes an annotation. Query-based annotations hold a reference to a Data View.

Both flavours are grouped inside an `event-annotation-group` object, which can be reused
across multiple Lens charts. The group itself references the Data View used for
query-based annotations (if any); manual-only groups have no external references.

From a restore standpoint, event annotation groups behave like any other Lens dependency:
they must be restored before the Lens charts that reference them, and they in turn depend
on their Data View being present first. The feature-state restore handles this ordering
automatically.

To create an annotation group in the Lens editor: open a time series chart, click
**Annotations** in the layer panel, add a layer, and save the group to the library so it
can be shared with other charts.

In [ ]:
heading('4.16 Event Annotation Groups')

annotations = find_saved_objects('event-annotation-group')
console.print(f'  Found {len(annotations)} event-annotation-group object(s)')
if annotations:
    pp(annotations[0], 'event-annotation-group saved object')
    info('Key fields:')
    info('  annotations[] — array of annotation definitions')
    info('    type: manual | query')
    info('    manual: fixed timestamp, label, line style/colour')
    info('    query: ES query that resolves to event timestamps at render time')
    info('  dataViewSpec  — embedded data view spec (or reference to existing data view)')
    info('references      — link to index-pattern if using a referenced data view')
else:
    info('No annotation groups yet — create a Lens chart and add an event annotation to see this type.')

---
## 4.17 Cross-Type Restore Ordering

Kibana saved objects form a **directed acyclic dependency graph**. An object with entries
in its `references` array depends on all referenced objects existing before it can render
correctly. When Kibana performs a feature-state restore, it walks this graph and resolves
objects in dependency order — you never need to restore types one at a time.

```
Restore order (simplified):

  1. config, space, tag          (no dependencies)
  2. index-pattern               (no dependencies)
  3. query                       (no dependencies)
  4. search                      (depends on: index-pattern)
  5. visualization               (depends on: index-pattern, search)
  6. lens                        (depends on: index-pattern)
  7. map                         (depends on: index-pattern)
  8. canvas-workpad              (no external dependencies — self-contained)
  9. event-annotation-group      (depends on: index-pattern)
 10. action (connector)          (no dependencies)
 11. alert (rule)                (depends on: action)
 12. cases                       (depends on: action for connectors)
 13. cases-comments              (depends on: cases)
 14. cases-user-actions          (depends on: cases)
 15. dashboard                   (depends on: visualization, lens, map, search, tag)
 16. short-url                   (depends on: dashboard)
```

This ordering also explains why **partial restores are risky**. If you restore only a
dashboard without its referenced visualizations, the dashboard object will be written to
Kibana but its panels will render as errors. Kibana does not validate references at restore
time — it trusts you to restore a consistent set of objects.

The cells below demonstrate this with a full-state snapshot, a complete wipe of all Kibana
state, and a restore — then verify that object counts match before and after.

In [ ]:
heading('4.17 Full Kibana state snapshot → delete all → restore all')

# Take a comprehensive snapshot capturing everything we've built
snap_name = 'snap-full-kibana-state'
delete_snapshot_if_exists(es, SNAP_REPO, snap_name)
result = snapshot_kibana_state(es, SNAP_REPO, snap_name)
success(f'Full state snapshot: {result["state"]}')

# Count objects before delete
types_to_check = ['index-pattern', 'dashboard', 'lens', 'visualization', 'search',
                  'alert', 'action', 'tag', 'query', 'cases']
before_counts = {}
for t in types_to_check:
    before_counts[t] = len(find_saved_objects(t))

t_table = Table('Type', 'Before', 'After restore')
for type_name, count in before_counts.items():
    t_table.add_row(type_name, str(count), '...')
console.print(t_table)

In [ ]:
# Delete all Kibana sample data (this removes all saved objects from sample datasets)
for dataset in ['ecommerce', 'flights', 'logs']:
    try:
        remove_sample_data(dataset)
    except Exception as e:
        warn(f'Could not remove {dataset}: {e}')

# Also delete our training objects
for obj_type, obj_id in [
    ('tag', tag_id),
    ('alert', rule_id),
    ('action', connector_id),
]:
    try:
        kibana_delete(f'/api/saved_objects/{obj_type}/{obj_id}')
    except Exception:
        pass

time.sleep(2)

after_delete = {}
for t in types_to_check:
    after_delete[t] = len(find_saved_objects(t))

t2 = Table('Type', 'Before delete', 'After delete')
for type_name in types_to_check:
    t2.add_row(type_name, str(before_counts[type_name]), str(after_delete[type_name]))
console.print(t2)

In [ ]:
# Now restore from snapshot
heading('Restoring full Kibana state')
restore_kibana_state(es, SNAP_REPO, snap_name)
time.sleep(6)  # Give Kibana time to process the restored .kibana* indices

# Count after restore
after_restore = {}
for t in types_to_check:
    after_restore[t] = len(find_saved_objects(t))

t3 = Table('Type', 'Before', 'After delete', 'After restore', 'Match?')
for type_name in types_to_check:
    before = before_counts[type_name]
    after = after_restore[type_name]
    match = '[green]✓[/green]' if before == after else '[red]✗[/red]'
    t3.add_row(type_name, str(before), str(after_delete[type_name]), str(after), match)
console.print(t3)

## Summary

| Object type | Has external references? | Secrets encrypted? |
|------------|--------------------------|-------------------|
| `index-pattern` | No | No |
| `search` | Yes → `index-pattern` | No |
| `visualization` | Yes → `index-pattern`, `search` | No |
| `lens` | Yes → `index-pattern` | No |
| `map` | Yes → `index-pattern` | No |
| `dashboard` | Yes → visualizations, lens, maps, searches, tags | No |
| `canvas-workpad` | No (self-contained) | No |
| `tag` | No | No |
| `query` | No | No |
| `space` | No | No |
| `alert` (rule) | Yes → `action` | No |
| `action` (connector) | No | **Yes** — re-enter after cross-cluster restore |
| `cases` | Yes → `action` | No |
| `config` | Yes → `index-pattern` (defaultIndex) | No |
| `short-url` | Yes → target object | No |
| `event-annotation-group` | Yes → `index-pattern` | No |

**Next:** [05_restoring_snapshots.ipynb](05_restoring_snapshots.ipynb)